# 日英音声データを作る — Kokoro・JParaCrawl / Colab L4

JParaCrawlの文章からKokoroで日英音声を作り、Whisperの文字起こしで品質を確認します。**L4 GPUを選び、上から1段階ずつ実行してください。**

| 段階 | やること | 次へ進む目安 |
| --- | --- | --- |
| 1〜2 準備 | Drive接続・環境構築・入力とGPUの確認 | エラーなく完了 |
| 3 少量テスト | 同じセルで10件、確認後に100件 | 音声を聴き、採用率・失敗理由を確認 |
| 4〜5 速度調整 | 並列数と文字起こしのバッチ数を比較・設定 | 推奨値と品質差を確認 |
| 6 最終テスト | 選んだ設定で1000件 | 音質・CER/WER・処理速度・容量を確認 |
| 7〜9 本番 | 進捗確認 → 生成 → 完成データの書き出し | summaryのcompleteがtrue |

**初期設定では10件テストまでです。** 速度測定・1000件テスト・本番は、それぞれ手動で有効にします。10件の結果を見るため、最初は「すべて実行」を使わず段階ごとに進めてください。

**再接続したときは、止まった段階に応じて次のセルだけ実行します。** 完了したテストのやり直しは不要です。

| 再開したい処理 | 実行するセル |
| --- | --- |
| 1000件テストの続き | **1 → 2 → 5 → 6**。5で設定を確認して `PERFORMANCE_CONFIG_CONFIRMED = True`、6で `RUN_BENCHMARK_1000 = True` |
| 本番の続き | **1 → 2 → 5 → 7 → 8**。8で `APPROVE_PRODUCTION = True`、残り全部なら `RUN_ALL_REMAINING = True` |
| 全shard完了後の書き出し | 本番再開の手順で完了を確認してから **9** |

設定・入力・保存先は以前と同じものを使います。音声・進捗・設定・ログはDriveに残り、環境とモデルのキャッシュはColabのローカルディスクにあります。速度測定中にローカル音声が消えた場合は4をやり直します。

このノートブックは `main` ブランチのコードを使います。Qwen版とは保存先が別です。同じ保存先を複数のランタイムから同時に実行しないでください。


## 1. Driveに接続し、実行環境を準備する

実行するとDriveの接続許可が表示され、その後コード・Python 3.12・音声処理ライブラリを準備します。初回はダウンロードがあります。最後に **「環境の準備が完了しました」** と表示されたら2へ進んでください。

Colabの「ランタイムのタイプを変更」で **L4 GPU** を選んでおきます。このセルは再接続時にも実行します。既存のコードに未保存の変更がある場合は停止します。

手動パッチ適用後のランタイムで「未コミット変更」のため停止した場合は、このセルの `REPO_DIR` を `/content/ja-en-direct-s2st-corpus-kokoro-v2` などの新しい名前に変えて再実行してください。コードを別フォルダへ取得し、Drive上のデータ・設定・進捗は引き続き同じものを使います。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json
from pathlib import Path
REPO_DIR = Path('/content/ja-en-direct-s2st-corpus-kokoro')
BRANCH = 'main'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, 'https://github.com/Yaaamashiro/ja-en-direct-s2st-corpus.git', str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO_DIR, text=True)
    assert not dirty.strip(), '未コミット変更があります。保存してから再開してください。'
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'switch', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', 'origin/' + BRANCH], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'espeak-ng', 'build-essential', 'cmake'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.8.22'], check=True)
UV = [sys.executable, '-m', 'uv']
subprocess.run(UV + ['python', 'install', '3.12'], check=True)
ENV_DIR = Path('/content/s2st-kokoro-py312')
if not ENV_DIR.exists():
    subprocess.run(UV + ['venv', '--python', '3.12', '--seed', str(ENV_DIR)], check=True)
PYTHON = str(ENV_DIR / 'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', 'torch==2.7.1', 'torchaudio==2.7.1', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', 'requirements/kokoro-colab.txt'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'install', '-e', '.', '--no-deps'], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)
# UniDicパッケージだけでは辞書本体は入らないため、版を指定して取得
dictionary_check = "from pathlib import Path; import unidic; p=Path(unidic.DICDIR); assert (p/'sys.dic').is_file() and (p/'version').read_text().strip() == 'unidic-3.1.0+2021-08-31'"
if subprocess.run([PYTHON, '-c', dictionary_check]).returncode:
    subprocess.run([PYTHON, '-m', 'unidic', 'download', '3.1.0+2021-08-31'], check=True)
# Qwen用バッチ設定を引き継がない
os.environ.pop('S2ST_TTS_BATCH_SIZE', None)
os.environ.pop('S2ST_ASR_BATCH_SIZE', None)
os.environ['HF_HOME'] = '/content/hf-kokoro-cache'
os.environ['S2ST_INPUT_INDEX_DIR'] = '/content/s2st-input-index'
os.environ['S2ST_MODEL_ROOT'] = '/content/kokoro-models'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('環境の準備が完了しました。次は2の入力確認です。', flush=True)


## 2. 入力データ・保存先・GPUを確認する

Driveの設定ファイルを初回だけ作成し、入力の件数・重複・SHA256、GPUと容量を検査します。続いて音声モデル・辞書・声をダウンロードします。**ここでは音声を生成しません。**

入力は `jparacrawl-filtering/outputs/text-corpus/` の `manifest.jsonl` と `summary.json` です。まだない場合は、先にjparacrawl-filtering側の `scripts/export_text_corpus.py` で書き出してください。

- **設定：** Driveの `ja-en-direct-s2st-corpus/kokoro-jparacrawl-l4-config.yaml`。保存済みの設定は上書きしません。
- **ログ：** 同じフォルダの `kokoro-logs/`。失敗したときは、最後の例外だけでなく直前のログも確認します。
- **GPU：** 最低20 GiB。初期設定は音声生成の並列数1、Whisperのバッチ数8です。音声生成と文字起こしは順番に実行します。

パスや声を変える場合は、表示された設定ファイルを編集してこのセルを再実行してください。声・モデル・ASRバッチ数などを変えて再試験する場合は、以前の結果と混ざらないよう `smoke.output_dir` も新しい保存先にします。


In [ ]:
import shutil
DATA_ROOT = Path('/content/drive/MyDrive/ja-en-direct-s2st-corpus')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG = DATA_ROOT / 'kokoro-jparacrawl-l4-config.yaml'
if not CONFIG.exists():
    shutil.copyfile(REPO_DIR / 'configs/production-kokoro-jparacrawl-l4.yaml', CONFIG)
print(CONFIG.read_text())
# 保存済み設定からパスを取得（ホストPythonのyamlには依存しない）
def read_settings():
    script = 'import yaml,json,sys; print(json.dumps(yaml.safe_load(open(sys.argv[1]))))'
    return json.loads(subprocess.check_output([PYTHON, '-c', script, str(CONFIG)], text=True))
settings = read_settings()
for key in ('manifest', 'summary'):
    path = Path(settings['prepare'][key])
    assert path.is_file(), f'export入力がありません: {path}'
    print(key, path, path.stat().st_size)
# 実環境の依存・実装commitも保存
ENV_REPORT = DATA_ROOT / 'kokoro-environment'
ENV_REPORT.mkdir(exist_ok=True)
(ENV_REPORT / 'pip-freeze.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
(ENV_REPORT / 'git-commit.txt').write_text(subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True))

from datetime import datetime
LOG_DIR = DATA_ROOT / 'kokoro-logs'
LOG_DIR.mkdir(exist_ok=True)
def run_command(command, *args, production=False):
    cmd = [PYTHON, '-u', '-m', 's2st_corpus.cli', '--config', str(CONFIG)]
    if production:
        cmd.append('--allow-production')
    cmd.extend([command, *map(str, args)])
    log_path = LOG_DIR / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + command + '.log')
    print('Log:', log_path, flush=True)
    with log_path.open('w', encoding='utf-8') as log:
        with subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                rc = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if rc:
        raise RuntimeError(f'{command} failed ({rc}): {log_path}')

print('入力データを検証します。', flush=True)
run_command('prepare')
print('GPUと空き容量を確認します。', flush=True)
run_command('preflight')
print('音声モデル・辞書・声を準備します。', flush=True)
run_command('preload')
print('入力とGPUの確認が完了しました。次は3の10件テストです。', flush=True)


## 3. 10件を試し、確認できたら100件に増やす

まずそのまま実行します。日本語・英語の音声を作り、Whisperで文字起こしして品質を検査します。**10件の音声と結果を確認した後に、`SMOKE_PAIRS = 100` に変えて同じセルをもう一度実行してください。**

先頭10件と先頭100件を使い、件数ごとに別フォルダへ保存します。実行後は採用数・不採用数と、最初の採用ペアの音声を表示します。エラーなく終了しても全件採用とは限りません。不採用理由は結果フォルダの `all.jsonl` にあります。

声の候補も聴きたい場合だけ `RUN_VOICE_SAMPLES = True` にします。候補の試聴だけでは、生成に使う声は変わりません。


In [ ]:
SMOKE_PAIRS = 10  # 最初は10。結果を確認してから100に変えて再実行
RUN_SMOKE = True  # テスト済みで再実行しないときはFalse
RUN_VOICE_SAMPLES = False  # 声の候補も聴きたい場合だけTrue

VOICE_DIR = DATA_ROOT / 'kokoro-voice-samples'
if RUN_VOICE_SAMPLES:
    run_command('voice-samples', '--output-dir', VOICE_DIR, '--en-voices', 'af_heart', 'af_bella', 'am_adam')
    from IPython.display import Audio, display
    for path in sorted(VOICE_DIR.glob('*.wav')):
        print(path.name)
        display(Audio(filename=str(path)))

if RUN_SMOKE:
    assert SMOKE_PAIRS in (10, 100), 'このセルは10件または100件用です。1000件は6で実行します。'
    run_command('smoke-test', '--pairs', SMOKE_PAIRS)
    settings = read_settings()
    release_dir = Path(settings['smoke']['output_dir']) / f'pairs-{SMOKE_PAIRS}/production/manifests/releases'
    summary = json.loads((release_dir / 'summary.json').read_text(encoding='utf-8'))
    print('結果フォルダ:', release_dir)
    keys = ['total_pairs', 'accepted_pairs', 'rejected_pairs', 'acceptance_rate', 'failures', 'tts_rtf_total']
    print(json.dumps({key: summary.get(key) for key in keys}, ensure_ascii=False, indent=2))
    with (release_dir / 'accepted.jsonl').open(encoding='utf-8') as manifest:
        first_line = manifest.readline()
    if first_line:
        from IPython.display import Audio, display
        pair = json.loads(first_line)
        for lang in ('ja', 'en'):
            print(lang, pair[f'{lang}_text'])
            display(Audio(filename=pair[f'{lang}_wav_16k']))
    else:
        print('採用ペアがありません。all.jsonlで不採用理由を確認してください。')
else:
    print('少量テストはスキップしました。')


## 4. L4で速度を測り、並列数・バッチ数を選ぶ

100件の結果を確認したら `RUN_L4_TUNING = True` にして実行します。**この測定もCUを消費します。** 学習用200ペアで、次の2種類を順に比較します。

- **Kokoroの並列数（workers）：** 1 / 2 / 4 / 6。複数プロセスで音声を生成します。
- **Whisperのバッチ数：** 8 / 16 / 24 / 32。同じ音声を一度に何件文字起こしするかを変えます。

`TUNING_STAGE = 'all'` なら両方を続けて測定します。途中から文字起こしの測定だけをやり直す場合は `'asr'` を選べますが、同じランタイムの音声と同じコードのcommitが必要です。音声が消えたりコードを更新した場合は `'all'` でやり直します。

文の長さごとの希望配分は4・30・80・66・20件です。不足する区分がある場合は、ほかの区分へ再配分して総数200件を保ちます。実際の配分はログの `[selection]` とレポートの `selection` に残ります。全候補が同じ200ペアを使います。

最後に **推奨並列数・推奨バッチ数** を表示します。速度・品質の一致・VRAMの余裕から選ぶため、メモリ使用量が最大の設定が選ばれるとは限りません。推奨値が `None` の場合は候補の失敗理由を確認してください。設定ファイルには自動反映しません。


In [ ]:
RUN_L4_TUNING = False  # 100件テストを確認してからTrue
TUNING_STAGE = 'all'  # 'all': 両方 / 'tts': 音声生成だけ / 'asr': 文字起こしだけ

if RUN_L4_TUNING:
    assert TUNING_STAGE in ('all', 'tts', 'asr')
    run_command('preflight')
    run_command('benchmark-performance', '--stage', TUNING_STAGE)
else:
    print('速度測定はスキップしました。保存済みのレポートがあれば表示します。')

PERFORMANCE_REPORT = Path(read_settings().get('performance', {}).get('report_dir', str(DATA_ROOT / 'kokoro-performance'))) / 'l4-performance.json'
if PERFORMANCE_REPORT.is_file():
    performance_report = json.loads(PERFORMANCE_REPORT.read_text())
    print('GPU:', performance_report.get('gpu'))
    print('推奨する音声生成の並列数:', performance_report.get('recommended_tts_workers'))
    print('推奨する文字起こしのバッチ数:', performance_report.get('recommended_asr_batch_size'))
    for stage in ('tts_candidates', 'asr_candidates'):
        print('音声生成の比較:' if stage == 'tts_candidates' else '文字起こしの比較:')
        for candidate in performance_report.get(stage, []):
            keys = ['worker_count', 'batch_size', 'status', 'audio_hours_per_wall_hour', 'audio_seconds_per_wall_second', 'headroom_warning', 'quality_equivalent', 'gpu_peak_used_gib_sampled', 'peak_reserved_vram_gib_reference', 'error']
            print({key: candidate.get(key) for key in keys})
else:
    print('速度測定のレポートはまだありません。')

print('audio_hours_per_wall_hour / audio_seconds_per_wall_second: 大きいほど高速')
print('quality_equivalent: 品質比較の一致 / headroom_warning: VRAMの余裕が少ない候補')
print('gpu_peak_used_gib_sampled: GPU全体の使用量を5秒ごとに測定した最大値')
print('peak_reserved_vram_gib_reference: プロセスごとの参考値。GPU全体の実測ピークとは異なります。')


## 5. 選んだ設定を保存し、確認する

4の結果を見て、Driveの `kokoro-jparacrawl-l4-config.yaml` をColabのファイル一覧などから開き、次の2項目を編集・保存します。

| 項目 | 意味 |
| --- | --- |
| `tts.worker_count` | Kokoroで同時に動かすプロセス数 |
| `asr.batch_size` | Whisperで一度に処理する音声数 |

モデル・声・精度・QCの合格基準はそのままにします。下のセルを実行して保存した値を確認してください。値を確認できたら `PERFORMANCE_CONFIG_CONFIRMED = True` に変えて再実行し、6へ進みます。本番開始後は同じ設定と保存先を使って再開します。


In [ ]:
settings = read_settings()
print('Config:', CONFIG)
print('Kokoro workers:', settings['tts'].get('worker_count', 1))
print('Whisper batch:', settings['asr']['batch_size'])
PERFORMANCE_CONFIG_CONFIRMED = False
print('設定の確認:', '確認済み。6へ進めます。' if PERFORMANCE_CONFIG_CONFIRMED else '未確認。表示された設定を確認してください。')


## 6. 最終設定で1000件テストを行う

`RUN_BENCHMARK_1000 = True` にして実行します。5で設定を確認済みにする必要があります。学習用データから文の長さに応じて1000ペアを固定抽出し、音声生成から品質検査まで通します。**このテストもCUを消費します。**

希望配分は20・150・400・330・100件です。下の `BENCHMARK_SHORTFALL = 'redistribute'` は、不足分をほかの区分へ再配分して1000件を保つ指定です。配分不足をエラーにしたい場合は `'error'` にします。実配分は `input.selection.json` とsummaryに保存します。以前の1000件テストと配分が変わる場合は、2で `smoke.output_dir` を新しい保存先にしてから実行してください。

採用率・CER/WER・長文の失敗・処理速度・音質を確認し、本番へ進むか判断します。表示項目の見方は次のとおりです。

| 項目 | 意味 |
| --- | --- |
| `accepted_pairs / rejected_pairs` | 採用・不採用ペア数 |
| `attempt_metrics / failures` | CER/WERなどの品質指標と失敗の集計 |
| `duration_failures_by_language_and_length_bin` | 言語・文の長さ別の音声時間エラー |
| `tts_rtf_total` | 音声生成時間 ÷ 音声の長さ。小さいほど高速 |
| `phase_throughput` | 音声生成・文字起こしそれぞれの処理量と時間 |

結果は `smoke.output_dir/pairs-1000/production/manifests/releases/` に保存します。中断後に再開した場合、全体の所要時間が過小評価されることがあります。


In [ ]:
RUN_BENCHMARK_1000 = False
BENCHMARK_SHORTFALL = 'redistribute'  # 不足分を再配分。固定配分を必須にする場合は'error'
if RUN_BENCHMARK_1000:
    assert PERFORMANCE_CONFIG_CONFIRMED, '推奨値を確認し、設定を手動で確定してから実行してください。'
    run_command('smoke-test', '--pairs', 1000, '--benchmark-seed', 42,
                '--benchmark-shortfall', BENCHMARK_SHORTFALL)
    settings = read_settings()
    report_path = Path(settings['smoke']['output_dir']) / 'pairs-1000/production/manifests/releases/summary.json'
    report = json.loads(report_path.read_text())
    keys = ['total_pairs', 'length_bin_distribution', 'accepted_pairs', 'rejected_pairs', 'ja_hours', 'en_hours', 'attempt_metrics', 'failures', 'duration_failures_by_length_bin', 'duration_failures_by_language_and_length_bin', 'tts_rtf_total', 'asr_phase_wall_seconds', 'asr_inference_seconds', 'full_pipeline_wall_seconds', 'checkpoint_writes', 'checkpoint_write_seconds', 'directory_scan_count', 'directory_scan_seconds', 'train', 'dev', 'test']
    view = {key: report.get(key) for key in keys}
    view['gpu'] = [i.get('gpu') for i in report['invocations']]
    view['parent_peak_allocated_vram_gib'] = max((i.get('peak_allocated_vram_gib', 0) for i in report['invocations']), default=0)
    view['parent_peak_reserved_vram_gib'] = max((i.get('peak_reserved_vram_gib', 0) for i in report['invocations']), default=0)
    view['tts_worker_per_process_peaks'] = [i.get('tts_worker_per_process_peak', {}) for i in report['invocations']]
    view['phase_throughput'] = [{key: i.get(key) for key in ('kokoro_worker_count', 'asr_batch_size', 'tts_generated_utterances', 'tts_generated_audio_seconds', 'tts_phase_seconds', 'tts_audio_hours_per_wall_hour', 'asr_audio_seconds', 'asr_phase_seconds')} for i in report['invocations']]
    print(json.dumps(view, ensure_ascii=False, indent=2))
else:
    print('1000件テストは無効です。5の設定確認後に有効化してください。')


## 7. 本番の進捗と見積もりを確認する

生成は行いません。初回は処理計画を確認し、再接続したときは残りのshardを確認します。**shardはデータを分割した処理単位**です。

`completed_shards` が完了数、`remaining_shards` が残り、`next_shard` が次の処理対象です。設定の `expected_hours_per_language` は容量見積もりに使う想定値で、実際にできる音声時間を保証する値ではありません。


In [ ]:
run_command('plan')
run_command('status')


## 8. 本番の音声生成を実行・再開する

1000件の品質・速度・容量を確認して本番へ進むと決めたら、`APPROVE_PRODUCTION = True` にします。

- **まず1 shard：** `RUN_ALL_REMAINING = False` のまま実行します。
- **残りをすべて処理：** 最初のshardの結果を確認後、`RUN_ALL_REMAINING = True` に変えて再実行します。
- **中断後の再開：** 1〜2 → 5（保存済みの設定を確認）→ 7 → このセルの順で実行します。保存済みの進捗から続けます。

音声と進捗はDriveの `run.output_dir` に保存します。処理ログは2で表示した `kokoro-logs/` に残ります。本番の終了後は9へ進みます。


In [ ]:
APPROVE_PRODUCTION = False  # 1000件の結果を確認し、本番を始める場合だけTrue
RUN_ALL_REMAINING = False  # False: 次の1 shard / True: 残りすべて

if APPROVE_PRODUCTION:
    assert PERFORMANCE_CONFIG_CONFIRMED, '5で使用する設定を確認してください。'
    command = 'run-remaining-shards' if RUN_ALL_REMAINING else 'run-next-shard'
    run_command(command, production=True)
    run_command('status')
else:
    print('本番は無効です。1000件テストの結果を確認してから有効化してください。')


## 9. 完成データを書き出し、保存先を確認する

全shardが完了した後、`WRITE_RELEASE = True` にして実行します。元の入力順で最終manifestを書き出します。未完了のshardがある場合はエラーになります。途中集計だけが必要な場合は、別途 `consolidate --allow-incomplete` を使います。

既定の `'full'` は保存したWAVを開き、形式・長さ・SHA256を照合します。Drive上で時間がかかる場合は `RELEASE_AUDIO_VERIFICATION = 'inventory'` を選べます。このモードは保存済みのQC記録とWAVの存在を検査し、音声ファイルの中身は再照合しません。使用したモードはsummaryの `audio_verification` に記録します。

保存先は **`run.output_dir/manifests/releases/`** です。

| ファイル | 内容・使い道 |
| --- | --- |
| `accepted.jsonl` | QC合格ペアの全split。**alignment-transfer-s2utへ渡すmanifest** |
| `train.jsonl / dev.jsonl / test.jsonl` | 各splitのQC合格ペアを個別に確認・利用するためのファイル |
| `all.jsonl` | 不採用も含む全ペアの結果 |
| `summary.json` | 件数・採用率・音声時間など。最後に公開される完了確認用ファイル |

`summary.json` の **`complete: true` と `missing_shards: []`** を確認してください。alignment-transfer-s2utはtrain/devを同じmanifest内に必要とするため、`accepted.jsonl` を使います。


In [ ]:
WRITE_RELEASE = False  # 全shardの完了を確認したらTrue
RELEASE_AUDIO_VERIFICATION = 'full'  # 'inventory' でファイルの存在とQC記録を確認
if WRITE_RELEASE:
    run_command('consolidate', '--audio-verification', RELEASE_AUDIO_VERIFICATION)
    release_dir = Path(read_settings()['run']['output_dir']) / 'manifests/releases'
    summary = json.loads((release_dir / 'summary.json').read_text(encoding='utf-8'))
    print('完成データの保存先:', release_dir)
    print('下流に渡すmanifest:', release_dir / 'accepted.jsonl')
    print(json.dumps(summary, ensure_ascii=False, indent=2))
else:
    print('本番データの書き出しは無効です。テスト結果は各pairs-Nフォルダにあります。')
